# Quick Test: v4+OldCE với TOP_K=100

**Câu hỏi:** Tăng TOP_K từ 50 → 100 có giúp Recall@1 không?

Dùng lại v4 FAISS + OldCE, không cần train lại.

In [1]:
import json, csv
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder

ROOT     = Path(".")
EVAL_DIR = ROOT / "outputs" / "eval"
TMP_DIR  = ROOT / "outputs" / "tmp"
MDL_DIR  = ROOT / "outputs" / "models"

FT_BI_PATH   = MDL_DIR / "legal_hf_finetuned" / "final"
FAISS_V4     = TMP_DIR / "faiss_v4.index"
MAP_V4       = TMP_DIR / "faiss_mapping_v4.jsonl"
EVAL_QA_FILE = EVAL_DIR / "eval_qa.jsonl"

# CE v5 path (D_skip14 — best CE)
CE_V5_PATH = MDL_DIR / "cross_encoder_v5fix" / "saved_model"
if not CE_V5_PATH.exists(): CE_V5_PATH = MDL_DIR / "cross_encoder_v5fix"

# ── TEST CONFIGS ──
TOP_K_LIST = [30, 50, 100]   # Grid test 3 values cùng lúc
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_jsonl(path):
    rows, err = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: err+=1
    return rows

def is_hit(fid, ec, mapping):
    row=mapping[fid]
    for e in ec:
        ci=e.get("chunk_index",-2)
        if ci!=-1 and row["chunk_index"]==ci: return True
        if row["van_ban"]==e.get("van_ban","") and row["dieu"]==e.get("dieu","") and row["khoan"]==e.get("khoan",""): return True
    return False

def avg(lst): return round(sum(lst)/len(lst),4) if lst else 0.0

print(f"Device: {DEVICE} | Testing TOP_K: {TOP_K_LIST}")

d:\SGU\CNTT\NCKH2025_2026\ChatBot\cross-encoder\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda | Testing TOP_K: [30, 50, 100]


In [2]:
print("Loading models...")
bi_model   = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
eval_qa    = load_jsonl(EVAL_QA_FILE)
ce_v5      = CrossEncoder(str(CE_V5_PATH), max_length=256, device=DEVICE)
print(f"Loaded ✓ | {index_v4.ntotal} vectors | {len(eval_qa)} questions")

# Pre-encode tất cả queries 1 lần
queries = [item["query"] for item in eval_qa]
q_embs  = bi_model.encode(queries, normalize_embeddings=True,
                           convert_to_numpy=True, batch_size=32,
                           show_progress_bar=True).astype("float32")
print(f"Query embeddings: {q_embs.shape}")

Loading models...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 745.56it/s, Materializing param=classifier.weight]                                    


Loaded ✓ | 1861 vectors | 323 questions


Batches: 100%|██████████| 11/11 [00:01<00:00,  9.52it/s]

Query embeddings: (323, 768)


In [ ]:
# Test tất cả TOP_K values
results = {}

for top_k in TOP_K_LIST:
    r_base   = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}
    r_rerank = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

    # Retrieve top_k cho tất cả queries cùng lúc
    _, all_ids = index_v4.search(q_embs, top_k)

    for i, (item, ids) in enumerate(tqdm(zip(eval_qa, all_ids),
                                         total=len(eval_qa),
                                         desc=f"TOP_K={top_k}")):
        query = item["query"]; ec = item["expected_citations"]
        ids   = ids.tolist()

        # Bi-encoder baseline
        for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
            r_base[key].append(1 if any(is_hit(fid,ec,mapping_v4) for fid in ids[:k] if fid>=0) else 0)
        mrr=0.0
        for rank,fid in enumerate(ids[:10],1):
            if fid>=0 and is_hit(fid,ec,mapping_v4): mrr=1.0/rank; break
        r_base["MRR@10"].append(mrr)

        # CE rerank
        cands  = [(mapping_v4[fid]["passage"],fid) for fid in ids if fid>=0]
        rscore = ce_v5.predict([[query,c[0]] for c in cands],batch_size=32) if cands else []
        ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
        r_ids  = [x[1] for x in ranked]

        for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
            r_rerank[key].append(1 if any(is_hit(fid,ec,mapping_v4) for fid in r_ids[:k]) else 0)
        mrr=0.0
        for rank,fid in enumerate(r_ids[:10],1):
            if is_hit(fid,ec,mapping_v4): mrr=1.0/rank; break
        r_rerank["MRR@10"].append(mrr)

    results[top_k] = {"base": {m:avg(v) for m,v in r_base.items()},
                      "rerank": {m:avg(v) for m,v in r_rerank.items()}}

# Print kết quả
print("\n" + "="*90)
print(f"  {'Config':<20} {'R@1':>8} {'R@3':>8} {'R@5':>8} {'MRR':>8}")
print("="*90)

V5_CE = {"R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307}

for top_k in TOP_K_LIST:
    base   = results[top_k]["base"]
    rerank = results[top_k]["rera nk"]
    print(f"  v4 base (k={top_k}){'':<6} {base['R@1']:>8.4f} {base['R@3']:>8.4f} {base['R@5']:>8.4f} {base['MRR@10']:>8.4f}")
    print(f"  v4+CE   (k={top_k}){'':<6} {rerank['R@1']:>8.4f} {rerank['R@3']:>8.4f} {rerank['R@5']:>8.4f} {rerank['MRR@10']:>8.4f}")
    print("  " + "-"*70)

print(f"  {'v5+CE (k=50)':<20} {V5_CE['R@1']:>8.4f} {V5_CE['R@3']:>8.4f} {V5_CE['R@5']:>8.4f} {V5_CE['MRR@10']:>8.4f}  ← best so far")
print("="*90)

# Lưu CSV
rows = []
for top_k in TOP_K_LIST:
    rows.append({"config": f"v4_base_k{top_k}", **results[top_k]["base"]})
    rows.append({"config": f"v4_CE_k{top_k}",   **results[top_k]["rerank"]})
rows.append({"config":"v5_CE_k50", **V5_CE})

out = EVAL_DIR / "topk_grid.csv"
with open(out,"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f,fieldnames=["config","R@1","R@3","R@5","MRR@10"])
    w.writeheader(); w.writerows(rows)
print(f"Saved → {out} ✓")

TOP_K=100: 100%|██████████| 323/323 [01:14<00:00,  4.34it/s]


  Config                    R@1      R@3      R@5      MRR
  v4 base (k=30)         0.5232   0.6594   0.7337   0.6091
  v4+CE   (k=30)         0.5418   0.6873   0.7245   0.6312
  ----------------------------------------------------------------------
  v4 base (k=50)         0.5232   0.6594   0.7337   0.6091
  v4+CE   (k=50)         0.5418   0.6873   0.7245   0.6307
  ----------------------------------------------------------------------
  v4 base (k=100)         0.5232   0.6594   0.7337   0.6091
  v4+CE   (k=100)         0.5387   0.6873   0.7276   0.6270
  ----------------------------------------------------------------------
  v5+CE (k=50)           0.5418   0.6873   0.7245   0.6307  ← best so far
Saved → outputs\eval\topk_grid.csv ✓
